# H&M 2년 M4 K=1 실제 q_C·순열 대조군 — replication seed 43

완료된 seed 42와 같은 동결 사양을 seed 43에서 반복합니다. 세 arm은 **M1(K=1) → 실제 q_C M4(K=1) → degree-matched q_C 순열 M4(K=1)** 순서로 학습합니다. 최종 test나 holdout은 만들지 않습니다. seed 44 노트북과 별도 Colab 런타임에서 병렬 실행할 수 있습니다.

## 런타임 중단 대응

- 각 arm의 **매 epoch가 끝날 때마다** 모델·optimizer·난수상태를 Google Drive에 원자적으로 저장합니다.
- 런타임이 끊기면 Drive를 다시 마운트하고 아래 셀을 같은 순서로 재실행하세요. 진행 중이던 arm은 **마지막으로 완료된 epoch 다음**부터 재개됩니다.
- 끊긴 시점에 실행 중이던 epoch 하나는 다시 수행될 수 있지만, 그 이전 완료 epoch는 보존됩니다.
- 이미 100 epoch를 마친 arm은 최종 checkpoint와 결과를 그대로 재사용하므로 다시 학습하지 않습니다.
- 저장 위치: `MyDrive/논문/data/results_v3_hm_m4_k1_assignment_control_hm2y_development_screen_v1`


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
from pathlib import Path
import subprocess
import sys

REVIEWED_SHA = '3188b01c360d54e78338eaebdf2a8ecb6a6d52d8'
REPO_URL = 'https://github.com/jung-un/clv-m2-lightgcn-runner.git'
REPO_DIR = Path('/content/clv-m2-lightgcn-runner-hm-m4-k1')

if REPO_DIR.exists() and not (REPO_DIR / '.git').exists():
    raise RuntimeError(f'기존 경로가 Git 저장소가 아닙니다: {REPO_DIR}')
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', '-q', REPO_URL, str(REPO_DIR)], check=True)
subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', '-q', 'origin', REVIEWED_SHA], check=True)
subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', '-q', '--detach', REVIEWED_SHA], check=True)
actual_sha = subprocess.check_output(['git', '-C', str(REPO_DIR), 'rev-parse', 'HEAD'], text=True).strip()
assert actual_sha == REVIEWED_SHA
os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
print('검토된 코드:', actual_sha)


In [ ]:
import json
import torch
import lightgcn_clv_m4_k1_assignment_control_hm2y as hm_screen

assert torch.cuda.is_available(), '런타임 유형에서 GPU를 선택하세요.'
assert hm_screen.CODE_VERSION == 'm4-personalized-positive-weight-k1-assignment-control-hm2y-development-screen-v1'
cfg = hm_screen.configure_hm2y_m4_assignment_screen(seed=43, shuffle_seed=43)
summary = hm_screen.preflight_summary(cfg)
assert summary['dataset'] == 'hm'
assert summary['seed'] == 43 and cfg.shuffle_seed == 43
assert summary['staged_replication']['seeds'] == [42, 43, 44]
assert summary['trained_models'] == list(hm_screen.MODEL_IDS)
assert summary['fixed']['negative_count'] == 1
assert summary['fixed']['batch_size'] == 131072
assert summary['fixed']['final_test_constructed'] is False
assert summary['fixed']['holdout_constructed'] is False
assert summary['checkpointing']['save_after_each_completed_epoch'] is True
print(json.dumps(summary, ensure_ascii=False, indent=2))
print('\n중단 후 재실행하면 마지막으로 완료된 epoch 다음부터 이어집니다.')


In [ ]:
# 이 셀은 한 번만 실행합니다. 런타임이 끊긴 경우에는 위 셀부터 다시 실행해도 안전합니다.
result_df = hm_screen.run_hm2y_m4_assignment_screen(cfg)


In [ ]:
from IPython.display import display

def show(frame):
    view = frame.copy()
    view.attrs = {}
    display(view)

core = [
    'model_id', 'm4_assignment', 'row_weight_cv',
    'recall@10', 'ndcg@10', 'recall@20', 'ndcg@20',
    'recall@50', 'ndcg@50',
    'price_purchase_amount_weighted_hit@10', 'vndcg@10',
    'coverage@10', 'user_value_tendency_recommended_price_alignment',
]
print('1) 핵심 절대지표')
show(result_df[[column for column in core if column in result_df.columns]])
print('2) 전체 비교지표')
show(result_df.attrs['comparison'])
print('3) Top-10 변경 비율')
show(result_df.attrs['top10_overlap'])
print('4) 사전 고정 판독')
print(json.dumps(result_df.attrs['decision'], ensure_ascii=False, indent=2))
print('5) 저장 파일')
print(json.dumps(result_df.attrs['result_paths'], ensure_ascii=False, indent=2))
